# 04: Logistic Regression - From Numbers to Decisions

## The Problem with Linear Regression for Classification

In the last notebook, we predicted continuous values (house prices). But what if we want to predict categories?

- Is this email **spam** or **not spam**?
- Is this review **positive** or **negative**?
- Will this customer **buy** or **not buy**?

This is **classification** - predicting discrete categories.

### The Web Dev Analogy

Think of it like form validation:
- Linear regression: "Score this from 1-100"
- Classification: "Valid or Invalid? True or False?"

We need to output **probabilities** (0 to 1), not unbounded numbers!

## What You'll Learn
- [ ] Explain why linear regression fails for classification tasks
- [ ] Implement the sigmoid function and logistic regression from scratch
- [ ] Evaluate classifiers using precision, recall, and F1 score

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 3**: `y = wx + b` and gradient descent | Same formula! We just wrap it in sigmoid to get probabilities (0 to 1) |
| **Lesson 3**: MSE loss | Classification needs a different loss — cross-entropy — that penalizes confident wrong predictions |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to make decisions! 🎯")

## 1. Why Linear Regression Fails for Classification

Let's see the problem:

In [ ]:
# Fake data: Hours studied vs Pass/Fail
hours_studied = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
passed = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])  # 0 = fail, 1 = pass

# Fit a line (linear regression)
from numpy.polynomial import polynomial as P
coef = np.polyfit(hours_studied, passed, 1)  # Degree 1 = line
linear_pred = np.polyval(coef, hours_studied)

# Plot
plt.figure(figsize=(10, 5))
plt.scatter(hours_studied, passed, s=100, c=['red' if p == 0 else 'green' for p in passed],
            edgecolors='black', zorder=3)

x_line = np.linspace(0, 12, 100)
y_line = np.polyval(coef, x_line)
plt.plot(x_line, y_line, 'b--', label='Linear regression')

# Show the problems
plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.axhline(1, color='gray', linestyle=':', alpha=0.5)
plt.fill_between(x_line, 1, 1.5, alpha=0.2, color='red', label='Invalid! (> 1)')
plt.fill_between(x_line, -0.5, 0, alpha=0.2, color='red', label='Invalid! (< 0)')

plt.xlabel('Hours Studied')
plt.ylabel('Probability of Passing')
plt.title('Problem: Linear Regression for Classification')
plt.ylim(-0.3, 1.3)
plt.legend()
plt.show()

print("\n❌ Problems with linear regression:")
print("   1. Predictions can be > 1 or < 0 (not valid probabilities!)")
print("   2. The decision boundary isn't clear")

## 2. The Sigmoid Function: Squishing to 0-1

We need a function that:
- Takes any number as input
- Outputs a value between 0 and 1 (a probability!)

Enter the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

It "squishes" any number to the range (0, 1).

In [ ]:
def sigmoid(z):
    """The sigmoid function - squishes any number to (0, 1)."""
    return 1 / (1 + np.exp(-z))

# Visualize sigmoid
z = np.linspace(-10, 10, 200)
s = sigmoid(z)

plt.figure(figsize=(10, 5))
plt.plot(z, s, 'b-', linewidth=2)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Decision threshold (0.5)')
plt.axhline(0, color='gray', linestyle=':', alpha=0.3)
plt.axhline(1, color='gray', linestyle=':', alpha=0.3)

# Annotations
plt.annotate('Large negative → ~0', xy=(-8, 0.001), fontsize=11, color='blue')
plt.annotate('Large positive → ~1', xy=(4, 0.98), fontsize=11, color='blue')
plt.annotate('z=0 → 0.5', xy=(0.5, 0.5), fontsize=11, color='red')

plt.xlabel('z (input)')
plt.ylabel('σ(z) (output)')
plt.title('The Sigmoid Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n📊 Key properties of sigmoid:")
print(f"   sigmoid(-10) = {sigmoid(-10):.6f}  (very close to 0)")
print(f"   sigmoid(0)   = {sigmoid(0):.6f}   (exactly 0.5)")
print(f"   sigmoid(10)  = {sigmoid(10):.6f}  (very close to 1)")

## 3. Logistic Regression: Linear + Sigmoid

Logistic regression combines:
1. **Linear function**: $z = w \cdot x + b$
2. **Sigmoid**: $\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$

So our prediction is:
$$\hat{y} = \sigma(w \cdot x + b)$$

This gives us a **probability** that we can threshold to make a decision!

In [ ]:
# Apply sigmoid to our linear model
z_line = np.polyval(coef, x_line)  # Linear combination
prob_line = sigmoid(z_line * 5)    # Scale and apply sigmoid (the *5 helps visualization)

plt.figure(figsize=(10, 5))
plt.scatter(hours_studied, passed, s=100, c=['red' if p == 0 else 'green' for p in passed],
            edgecolors='black', zorder=3)

# Linear (bad)
plt.plot(x_line, y_line, 'b--', alpha=0.5, label='Linear (bad)')

# Logistic (good!)
# Better fit with proper parameters
x_fine = np.linspace(0, 12, 200)
prob_logistic = sigmoid(2 * (x_fine - 4.5))  # Manually tuned for visualization
plt.plot(x_fine, prob_logistic, 'r-', linewidth=2, label='Logistic (good!)')

plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.axvline(4.5, color='gray', linestyle=':', alpha=0.5, label='Decision boundary')

plt.xlabel('Hours Studied')
plt.ylabel('Probability of Passing')
plt.title('Logistic Regression: S-Shaped Curve')
plt.ylim(-0.1, 1.1)
plt.legend()
plt.show()

print("\n✅ Logistic regression:")
print("   - Output is always between 0 and 1")
print("   - Clear decision boundary (where probability = 0.5)")
print("   - Can interpret output as probability!")

## 4. The Binary Cross-Entropy Loss

For classification, we use a different loss function called **binary cross-entropy** (BCE):

$$\text{BCE} = -\frac{1}{n}\sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)]$$

**Intuition**: 
- If true label is 1, penalize low predictions (want $\hat{y}$ close to 1)
- If true label is 0, penalize high predictions (want $\hat{y}$ close to 0)

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """
    Binary Cross-Entropy Loss.
    
    Small epsilon to avoid log(0).
    """
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

# Visualize the loss for a single example
predictions = np.linspace(0.01, 0.99, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# When true label = 1
loss_when_1 = -np.log(predictions)
axes[0].plot(predictions, loss_when_1, 'b-', linewidth=2)
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Loss')
axes[0].set_title('When True Label = 1\n(Want high predictions)')
axes[0].axvline(1, color='green', linestyle='--', alpha=0.5)

# When true label = 0
loss_when_0 = -np.log(1 - predictions)
axes[1].plot(predictions, loss_when_0, 'r-', linewidth=2)
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Loss')
axes[1].set_title('When True Label = 0\n(Want low predictions)')
axes[1].axvline(0, color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n🔑 Key insight:")
print("   - Loss explodes as prediction gets wrong!")
print("   - This strongly penalizes confident wrong predictions.")

## 5. Training Logistic Regression

Let's build and train a complete logistic regression model!

In [ ]:
# Generate more realistic data
np.random.seed(42)

# Two classes of points
n_per_class = 50

# Class 0: centered around (-1, -1)
class_0 = np.random.randn(n_per_class, 2) * 0.8 + np.array([-1, -1])

# Class 1: centered around (1, 1)
class_1 = np.random.randn(n_per_class, 2) * 0.8 + np.array([1, 1])

# Combine
X = np.vstack([class_0, class_1])
y = np.array([0] * n_per_class + [1] * n_per_class)

# Shuffle
shuffle_idx = np.random.permutation(len(X))
X, y = X[shuffle_idx], y[shuffle_idx]

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class 0', alpha=0.6, s=50)
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', alpha=0.6, s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Binary Classification Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: {X.shape} (samples × features)")
print(f"Labels shape: {y.shape}")

In [ ]:
class LogisticRegression:
    """
    Logistic Regression from scratch!
    """
    
    def __init__(self, learning_rate=0.1):
        self.lr = learning_rate
        self.weights = None
        self.bias = None
        self.history = {'loss': []}
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def predict_proba(self, X):
        """Predict probabilities."""
        z = np.dot(X, self.weights) + self.bias
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """Predict classes (0 or 1)."""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def compute_loss(self, y_true, y_pred):
        """Binary cross-entropy loss."""
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X, y, n_iterations=1000, verbose=True):
        """Train the model using gradient descent."""
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(n_iterations):
            # Forward pass
            y_pred = self.predict_proba(X)
            
            # Compute loss
            loss = self.compute_loss(y, y_pred)
            self.history['loss'].append(loss)
            
            # Compute gradients
            errors = y_pred - y
            d_weights = (1 / n_samples) * np.dot(X.T, errors)
            d_bias = (1 / n_samples) * np.sum(errors)
            
            # Update parameters
            self.weights -= self.lr * d_weights
            self.bias -= self.lr * d_bias
            
            if verbose and i % 200 == 0:
                acc = np.mean(self.predict(X) == y)
                print(f"Iteration {i:4d}: Loss = {loss:.4f}, Accuracy = {acc:.2%}")
        
        return self

# Train the model!
print("Training Logistic Regression...\n")
model = LogisticRegression(learning_rate=0.5)
model.fit(X, y, n_iterations=1000)

# Final accuracy
final_acc = np.mean(model.predict(X) == y)
print(f"\n✅ Final accuracy: {final_acc:.2%}")

In [ ]:
# Visualize the decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(model.history['loss'])
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

# Decision boundary
ax = axes[1]

# Create a mesh grid
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                      np.linspace(y_min, y_max, 100))

# Predict on mesh
Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdBu', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

# Plot points
ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class 0', edgecolors='black', s=50)
ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', edgecolors='black', s=50)

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Decision Boundary (black line = 50% probability)')
ax.legend()

plt.tight_layout()
plt.show()

print("\n🎯 The model learned a linear decision boundary!")
print(f"   Weights: {model.weights}")
print(f"   Bias: {model.bias:.3f}")

## 6. Understanding Predictions

Let's interpret what the model has learned:

In [ ]:
# Make predictions on specific points
test_points = np.array([
    [-2, -2],   # Clearly class 0
    [0, 0],     # On the boundary
    [2, 2],     # Clearly class 1
    [-1, 1],    # Ambiguous
])

print("Predictions for test points:")
print("=" * 50)

for point in test_points:
    prob = model.predict_proba(point.reshape(1, -1))[0]
    pred_class = "Class 1" if prob >= 0.5 else "Class 0"
    confidence = prob if prob >= 0.5 else 1 - prob
    print(f"Point {point} → P(Class 1) = {prob:.3f} → {pred_class} (confidence: {confidence:.1%})")

## 7. Evaluation Metrics

Accuracy isn't everything! Let's look at other important metrics:

In [ ]:
def confusion_matrix(y_true, y_pred):
    """Calculate confusion matrix components."""
    TP = np.sum((y_true == 1) & (y_pred == 1))  # True Positives
    TN = np.sum((y_true == 0) & (y_pred == 0))  # True Negatives
    FP = np.sum((y_true == 0) & (y_pred == 1))  # False Positives
    FN = np.sum((y_true == 1) & (y_pred == 0))  # False Negatives
    return TP, TN, FP, FN

# Calculate metrics
y_pred = model.predict(X)
TP, TN, FP, FN = confusion_matrix(y, y_pred)

accuracy = (TP + TN) / len(y)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Classification Metrics:")
print("=" * 40)
print(f"Accuracy:  {accuracy:.2%}  (overall correctness)")
print(f"Precision: {precision:.2%}  (of predicted positives, how many are correct?)")
print(f"Recall:    {recall:.2%}  (of actual positives, how many did we find?)")
print(f"F1 Score:  {f1:.2%}  (harmonic mean of precision & recall)")

print(f"\nConfusion Matrix:")
print(f"  True Positives:  {TP}")
print(f"  True Negatives:  {TN}")
print(f"  False Positives: {FP} (Type I errors)")
print(f"  False Negatives: {FN} (Type II errors)")

In [ ]:
# Visualize confusion matrix
cm = np.array([[TN, FP], [FN, TP]])

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar(label='Count')

# Add text annotations
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max()/2 else 'black'
        plt.text(j, i, f'{cm[i, j]}', ha='center', va='center', color=color, fontsize=20)

plt.xticks([0, 1], ['Predicted 0', 'Predicted 1'])
plt.yticks([0, 1], ['Actual 0', 'Actual 1'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 📝 Check Your Understanding

1. Why do we need sigmoid for classification?
2. What's the output range of the sigmoid function?
3. Why use cross-entropy loss instead of MSE for classification?
4. What does precision measure? What does recall measure?
5. When would you care more about precision vs recall?

In [ ]:
# --- Exercise 1: Sigmoid Values ---
# Compute sigmoid for these three inputs. Use the sigmoid() function from this notebook.

# YOUR CODE HERE:
sig_0 = None     # sigmoid(0)
sig_5 = None     # sigmoid(5)
sig_neg5 = None  # sigmoid(-5)

# --- Check ---
assert sig_0 is not None, "Compute sigmoid(0)!"
assert abs(sig_0 - 0.5) < 0.001, f"sigmoid(0) should be 0.5, got {sig_0:.4f}"
assert abs(sig_5 - 0.9933) < 0.001, f"sigmoid(5) should be ~0.993, got {sig_5:.4f}"
assert abs(sig_neg5 - 0.0067) < 0.001, f"sigmoid(-5) should be ~0.007, got {sig_neg5:.4f}"
print("Exercise 1 passed! ✓")

# --- Exercise 2: Manual Classification ---
# Given weights=[1.5, -0.5], bias=0.1, classify the point [2, 1].
# Steps: compute z = w·x + b, apply sigmoid, threshold at 0.5
ex_weights = np.array([1.5, -0.5])
ex_point = np.array([2, 1])
ex_bias = 0.1

# YOUR CODE HERE:
z = None            # Compute linear combination
prob = None         # Apply sigmoid to z
prediction = None   # 1 if prob >= 0.5, else 0

# --- Check ---
assert z is not None and prob is not None and prediction is not None, "Fill in all three!"
assert abs(z - 2.6) < 0.001, f"z = 1.5×2 + (-0.5)×1 + 0.1 = 2.6, got {z}"
assert prediction == 1, f"sigmoid(2.6) ≈ 0.93 > 0.5, so prediction should be 1, got {prediction}"
print("Exercise 2 passed! ✓")

# --- Exercise 3: Precision & Recall ---
# Given these confusion matrix values, compute precision and recall.
TP, FP, FN, TN = 8, 2, 3, 7

# YOUR CODE HERE:
precision = None  # TP / (TP + FP)
recall = None     # TP / (TP + FN)

# --- Check ---
assert precision is not None and recall is not None, "Compute both!"
assert abs(precision - 0.8) < 0.001, f"Precision = 8/(8+2) = 0.8, got {precision:.3f}"
assert abs(recall - 0.727) < 0.01, f"Recall = 8/(8+3) ≈ 0.727, got {recall:.3f}"
print(f"Exercise 3 passed! ✓  (Precision={precision:.2f}, Recall={recall:.3f})")

print("\n🎉 All exercises passed!")

## 🎯 Summary

You learned:
- **Sigmoid function** squishes values to (0, 1) for probabilities
- **Logistic regression** = Linear + Sigmoid
- **Cross-entropy loss** penalizes confident wrong predictions
- **Decision boundary** is where probability = 0.5
- **Metrics**: Accuracy, precision, recall, F1 score

**Next up**: Building a sentiment classifier! →